In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2
import imutils
import matlab.engine
import random
from gtda.homology import CubicalPersistence
from gtda.diagrams import PersistenceLandscape
import plotly.express as px

In [2]:
# Start MATLAB
eng = matlab.engine.start_matlab()

In [3]:
def crop_img(img):
	"""
	Finds the extreme points on the image and crops the rectangular out of them
	"""
	gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
	gray = cv2.GaussianBlur(gray, (3, 3), 0)

	# threshold the image, then perform a series of erosions +
	# dilations to remove any small regions of noise
	thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
	thresh = cv2.erode(thresh, None, iterations=2)
	thresh = cv2.dilate(thresh, None, iterations=2)

	# find contours in thresholded image, then grab the largest one
	cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
	cnts = imutils.grab_contours(cnts)
	c = max(cnts, key=cv2.contourArea)

	# find the extreme points
	extLeft = tuple(c[c[:, :, 0].argmin()][0])
	extRight = tuple(c[c[:, :, 0].argmax()][0])
	extTop = tuple(c[c[:, :, 1].argmin()][0])
	extBot = tuple(c[c[:, :, 1].argmax()][0])
	ADD_PIXELS = 0
	new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()
	
	return new_img

In [ ]:
input_dir = "MRI/Training/glioma"
output_dir = "MRI/Training/glioma_preprocessed"
os.makedirs(output_dir, exist_ok=True)

IMG_SIZE = 256
for filename in os.listdir(input_dir):
    if filename.lower().endswith(".jpg"):
        image_path = os.path.join(input_dir, filename)
        image = cv2.imread(image_path)
        cropped = crop_img(image)
        resized = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, resized)

## Everything in above code cells is copied from the Skull Extracted MRI notebook.

### Below is the process to compute preprocessing (skull-stripping and CLAHE) and save the processed images in an output directory.

In [4]:
def complete_preprocessing(image_path):
    """
    Complete preprocessing for the given image path.
    """
    
    # Use current directory where notebook and .m/.mat files are located
    eng.cd(os.getcwd(), nargout=0)
    model_path = os.path.abspath("NIVE.mat")

    # Run the MATLAB function
    result = eng.nive_extract_brain(image_path, model_path)

    # Convert MATLAB array to NumPy
    # result is a 3D array in column-major (Fortran-style)
    shape = tuple(result.size)  # (height, width, channels)
    skull_np_array = np.array(result._data, dtype=np.uint8).reshape(shape, order='F')
    
    cropped_skull = crop_img(skull_np_array)
    resized_skull = cv2.resize(cropped_skull, (200, 200), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized_skull, cv2.COLOR_BGR2GRAY)
    
    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    clahe_skull = clahe.apply(gray)
    
    # Convert clahe_skull to a PIL image
    clahe_img = Image.fromarray(clahe_skull)
    
    return clahe_img

In [ ]:
new_input_dir = "MRI/Training/glioma_preprocessed"
new_output_dir = "MRI/Training/glioma_preprocessed_comp"
os.makedirs(new_output_dir, exist_ok=True)

sorted_filenames = sorted([f for f in os.listdir(new_input_dir) if f.lower().endswith(".jpg")])
for filename in sorted_filenames:
    image_path = os.path.join(new_input_dir, filename)

    try:
        processed_img = complete_preprocessing(image_path) 
        output_path = os.path.join(new_output_dir, filename)
        processed_img.save(output_path) 
        print(f"Saved: {output_path}")
    except Exception as e:
        print(f"Error processing {filename}: {e}")

#print("Done with first 10 sorted images.")

Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0000.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0001.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0002.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0003.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0004.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0005.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0006.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0007.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0008.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-glTr_0009.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0010.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0011.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0012.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0013.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0014.jpg
Saved: MRI/Training/glioma_preprocessed_comp/Tr-gl_0015.jpg
Saved: MRI/Training/